In [1]:
!pip install -U autogluon > /dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!mkdir data/
!cp /content/drive/MyDrive/Sales-pred/sazerac_sales_prepared.parquet data/
!ls data

sazerac_sales_prepared.parquet


In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import RobustScaler, StandardScaler

from typing import List, Dict
from tqdm import tqdm
from matplotlib import pyplot as plt
from pathlib import Path

In [8]:
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
import pandas as pd
import numpy as np

df = pd.read_parquet("data/sazerac_sales_prepared.parquet")

store_counts = df.groupby('store').size()
valid_stores = store_counts[store_counts >= 70].index
df = df[df['store'].isin(valid_stores)]

df = df[df['store'].isin(valid_stores)]
df['sale_dollars'] = np.where(df['sale_dollars'] < 0, 0, df['sale_dollars'])
df['sale_dollars'] = np.where(df['sale_dollars'] > df['sale_dollars'].quantile(0.999),
                             df['sale_dollars'].quantile(0.999),
                             df['sale_dollars'])

STATIC_CATEGORICAL = [
    "name", "address", "city", "zipcode", "county",
]

STATIC_NUM = [
    "lon", "lat",
]

DYNAMIC = [
    "sale_bottles", "sale_bottles_mean", "sale_bottles_median", "sale_bottles_min", "sale_bottles_max",
    "sale_dollars_mean", "sale_dollars_median", "sale_dollars_min", "sale_dollars_max",
    "sale_liters", "sale_liters_mean", "sale_liters_median", "sale_liters_min", "sale_liters_max",
    "transaction_count",
    "state_bottle_cost_mean", "state_bottle_cost_median", "state_bottle_cost_min", "state_bottle_cost_max",
    "state_bottle_retail_mean", "state_bottle_retail_median", "state_bottle_retail_min", "state_bottle_retail_max",
    "avg_price_per_bottle", "avg_price_per_liter", "profit_margin", "discount_factor",
    "unique_categories", "unique_items",
    "pack_mean", "pack_median", "pack_min", "pack_max", "pack_sum",
    "bottle_volume_ml_mean", "bottle_volume_ml_median", "bottle_volume_ml_min", "bottle_volume_ml_max", "bottle_volume_ml_sum",
    "days_since_prev_purchase",
    "day_of_week_sin", "day_of_week_cos", "day_of_month_sin", "day_of_month_cos",
    "month_sin", "month_cos", "quarter_sin", "quarter_cos", "week_of_year_sin", "week_of_year_cos",
    "year", "is_weekend", "is_holiday", "days_to_nearest_holiday",
    "prev_1_purchase_sale_dollars", "prev_2_purchase_sale_dollars", "prev_3_purchase_sale_dollars", "prev_4_purchase_sale_dollars",
    "store_avg_sales", "store_avg_transactions", "store_avg_items", "store_size",
    "city_avg_sales", "county_avg_sales", "store_to_city_sales_ratio", "store_to_county_sales_ratio",
]

for w in [2, 4, 8, 12, 30, 60, 90]:
    for stat in ["mean", "std", "max", "min", "median"]:
        DYNAMIC.append(f"hist_{stat}_{w}_purchases_sale_dollars")
    DYNAMIC.extend([
        f"purchase_momentum_{w}",
        f"purchase_momentum_pct_{w}",
        f"hist_avg_days_between_purchases_{w}",
    ])

TARGET = ["sale_dollars"]

class FeaturePreprocessor:
    def __init__(self, static_cat_cols: List[str], static_num_cols: List[str],
                 dynamic_cols: List[str], target_cols: List[str]):
        self.static_cat_cols = static_cat_cols
        self.static_num_cols = static_num_cols
        self.dynamic_cols = dynamic_cols
        self.target_cols = target_cols

        self.static_scaler = StandardScaler()
        self.dynamic_scaler = StandardScaler()
        self.target_scaler = StandardScaler()

        self.cat_encodings = {}
        self.padding_values = {}

    def fit(self, df: pd.DataFrame):
        for col in self.static_cat_cols:
            unique_vals = df[col].dropna().unique()
            self.cat_encodings[col] = {val: idx + 2 for idx, val in enumerate(unique_vals)}
            self.padding_values[col] = 0  # padding token

        # Fit scalers
        if self.static_num_cols:
            self.static_scaler.fit(df[self.static_num_cols])
        if self.dynamic_cols:
            self.dynamic_scaler.fit(df[self.dynamic_cols])
        if self.target_cols:
            self.target_scaler.fit(df[self.target_cols])

        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Transform categorical features
        for col in self.static_cat_cols:
            # Convert to numeric categories, using 1 for unknown values
            df[col] = df[col].map(lambda x: self.cat_encodings[col].get(x, 1))
            # Fill missing with padding token
            df[col] = df[col].fillna(self.padding_values[col])

        # Transform numerical features
        if self.static_num_cols:
            df[self.static_num_cols] = self.static_scaler.transform(df[self.static_num_cols])
            # Fill missing with 0 after scaling
            df[self.static_num_cols] = df[self.static_num_cols].fillna(0)

        if self.dynamic_cols:
            df[self.dynamic_cols] = self.dynamic_scaler.transform(df[self.dynamic_cols])
            # Fill missing with 0 after scaling
            df[self.dynamic_cols] = df[self.dynamic_cols].fillna(0)

        if self.target_cols:
            df[self.target_cols] = self.target_scaler.transform(df[self.target_cols])

        return df

    def inverse_transform_targets(self, y: np.ndarray) -> np.ndarray:
        return self.target_scaler.inverse_transform(y)

class FeaturePreprocessorChronos(FeaturePreprocessor):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.target_scaler = None

    def fit(self, df: pd.DataFrame):
        original_target_cols = self.target_cols
        self.target_cols = []

        super().fit(df)

        self.target_cols = original_target_cols
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Handle categorical features
        for col in self.static_cat_cols:
            df[col] = df[col].map(lambda x: self.cat_encodings[col].get(x, 1))
            df[col] = df[col].fillna(self.padding_values[col])

        # Transform numerical features (skip target)
        if self.static_num_cols:
            df[self.static_num_cols] = self.static_scaler.transform(df[self.static_num_cols])
            df[self.static_num_cols] = df[self.static_num_cols].fillna(0)

        if self.dynamic_cols:
            df[self.dynamic_cols] = self.dynamic_scaler.transform(df[self.dynamic_cols])
            df[self.dynamic_cols] = df[self.dynamic_cols].fillna(0)

        # Explicitly keep original target values
        if self.target_cols:
            df[self.target_cols] = df[self.target_cols].copy()

        return df

SEQ_LEN, HORIZON = 30, 30
BATCH_SIZE, LR, EPOCHS = 30, 1e-3, 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

store_ids = df['store'].unique()
np.random.shuffle(store_ids)

n = len(store_ids)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_df = df[df.store.isin(store_ids[:n_train])]
val_df   = df[df.store.isin(store_ids[n_train:n_train + n_val])]
test_df  = df[df.store.isin(store_ids[n_train + n_val:])]

print(f"Train stores: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Initialize preprocessor without target scaling
preprocessor = FeaturePreprocessorChronos(
    static_cat_cols=STATIC_CATEGORICAL,
    static_num_cols=STATIC_NUM,
    dynamic_cols=DYNAMIC,
    target_cols=TARGET  # Track target but don't scale
)

preprocessor.fit(train_df)

# Apply preprocessing (static/dynamic features scaled, target remains original)
transformed_df = preprocessor.transform(df)

# # Ensure static features are constant per store
# for col in STATIC_CATEGORICAL + STATIC_NUM:
#     transformed_df[col] = transformed_df.groupby('store')[col].transform('first')

# # Create TimeSeriesDataFrame for AutoGluon
# ts_df = TimeSeriesDataFrame.from_data_frame(
#     transformed_df.reset_index(),
#     id_column="store",
#     timestamp_column="date",
#     target_column="sale_dollars",
#     static_features=STATIC_CATEGORICAL + STATIC_NUM,
#     known_covariates_names=DYNAMIC
# )

transformed_df = transformed_df.rename(columns={
    "store": "item_id",
    "date": "timestamp",
    "sale_dollars": "target"
})

static_cols = ["item_id"] + STATIC_CATEGORICAL + STATIC_NUM
static_features_df = transformed_df[static_cols].groupby("item_id").first().reset_index()

# Verify the dataframe contains item_id
print("Static features columns:", static_features_df.columns.tolist())

# Create TimeSeriesDataFrame
ts_df = TimeSeriesDataFrame.from_data_frame(
    df=transformed_df.reset_index(),
    id_column="item_id",
    timestamp_column="timestamp",
    static_features_df=static_features_df
)

Train stores: 156701 | Val: 34009 | Test: 35567
Static features columns: ['item_id', 'name', 'address', 'city', 'zipcode', 'county', 'lon', 'lat']


In [9]:
ts_df = ts_df.rename(columns = {"target": "sale_dollars"})

In [ ]:
# Split data
prediction_length = 30
train_data, test_data = ts_df.train_test_split(prediction_length=prediction_length)

# Initialize and train predictor with Chronos-Bolt
predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="sale_dollars",
    known_covariates_names=DYNAMIC,
    freq="W"
).fit(
    train_data,
    presets="medium_quality",
    hyperparameters={
        "Chronos": {
            "model_path": "bolt_small",
            "fine_tune": True,
            "fine_tune_epochs": 20,
            "covariate_regressor": "CAT",
            "target_scaler": "standard",
        }
    },
    time_limit=3600,
)

Frequency 'W' stored as 'W-SUN'
Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/content/AutogluonModels/ag-20250506_192549'
=================== System Info ===================
AutoGluon Version:  1.3.0
Python Version:     3.11.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Mar 30 16:01:29 UTC 2025
CPU Count:          2
GPU Count:          1
Memory Avail:       6.70 GB / 12.67 GB (52.8%)
Disk Space Avail:   70.11 GB / 112.64 GB (62.2%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'W-SUN',
 'hyperparameters': {'Chronos': {'covariate_regressor': 'CAT',
                                 'fine_tune': True,
                                 'fine_tune_epochs': 20,
                                 'model_path': 'bolt_small',
                                 'target_scaler': 'standard'}},
 'known_covariates_names': ['sale_bottles',
    

In [ ]:
# Evaluate and make predictions
predictions = predictor.predict(test_data)
leaderboard = predictor.leaderboard(test_data)
print(leaderboard)